In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

In [ ]:


# Dicionário com os arquivos estratégicos 
arquivos = {
    "Ativo e Passivo (Mensal)": "inf_mensal_fii_ativo_passivo_2025.csv",
    "Complemento (Mensal)": "inf_mensal_fii_complemento_2025.csv",
    "Geral (Mensal)": "inf_mensal_fii_geral_2025.csv",
    "Desempenho Imóvel (Trimestral)": "inf_trimestral_fii_imovel_desempenho_2025.csv",
    "Base trimestral":"inf_trimestral_fii_imovel_2025.csv"
}


In [3]:

# 1. Carregar apenas as colunas que queremos
colunas_passivo = ['CNPJ_Fundo_Classe', 'Data_Referencia', 'Total_Passivo']
colunas_comp = ['CNPJ_Fundo_Classe', 'Data_Referencia', 'Valor_Ativo', 'Patrimonio_Liquido', 
                'Valor_Patrimonial_Cotas', 'Percentual_Dividend_Yield_Mes']
#1.1 Carregando os csvs
df_passivo = pd.read_csv('inf_mensal_fii_ativo_passivo_2025.csv', sep=';', encoding='latin1', usecols=colunas_passivo)
df_comp = pd.read_csv('inf_mensal_fii_complemento_2025.csv', sep=';', encoding='latin1', usecols=colunas_comp)

# 2. Converter datas e isolar para o mês mais recente (último mês disponível)
df_passivo['Data_Referencia'] = pd.to_datetime(df_passivo['Data_Referencia'])
df_comp['Data_Referencia'] = pd.to_datetime(df_comp['Data_Referencia'])
ultima_data = df_comp['Data_Referencia'].max()
df_passivo_atual = df_passivo[df_passivo['Data_Referencia'] == ultima_data]
df_comp_atual = df_comp[df_comp['Data_Referencia'] == ultima_data]

# 3. O Cruzamento (Inner Join usando CNPJ e Data)
df_micro = pd.merge(df_comp_atual, df_passivo_atual, on=['CNPJ_Fundo_Classe', 'Data_Referencia'], how='inner')

# 4. Cálculo do LTV e padronização dos nomes para o modelo
df_micro['LTV'] = df_micro['Total_Passivo'] / df_micro['Valor_Ativo']
df_micro.rename(columns={
    'Valor_Patrimonial_Cotas': 'VPA', 
    'Percentual_Dividend_Yield_Mes': 'DY_Mes'
}, inplace=True)

# Visualizando a Matriz de Fundamentos limpa
print("\n--- Base de Fundamentos Extraída ---")
print(df_micro[['CNPJ_Fundo_Classe', 'LTV', 'VPA', 'DY_Mes']].head())


--- Base de Fundamentos Extraída ---
    CNPJ_Fundo_Classe       LTV          VPA    DY_Mes
0  00.332.266/0001-31  0.001643    92.127221  0.000000
1  00.613.094/0001-74  1.127006   -32.997029 -0.000908
2  00.762.723/0001-28  0.016373    17.800085  0.000000
3  00.868.235/0001-08  0.020158  2239.480309  0.005884
4  01.201.140/0001-90  0.007323   110.796096  0.005895


In [4]:


# 1. CARREGAR E ISOLAR DADOS FÍSICOS (Trimestral)
colunas_imovel = ['CNPJ_Fundo_Classe', 'Data_Referencia', 'Percentual_Vacancia', 
                  'Percentual_Inadimplencia', 'Area']

# Lendo o CSV com as colunas corretas
df_imoveis = pd.read_csv('inf_trimestral_fii_imovel_2025.csv', sep=';', encoding='latin1', usecols=colunas_imovel)

# Isolando a "foto" mais recente do trimestre
df_imoveis['Data_Referencia'] = pd.to_datetime(df_imoveis['Data_Referencia'])
ultima_data_tri = df_imoveis['Data_Referencia'].max()
df_imoveis_atual = df_imoveis[df_imoveis['Data_Referencia'] == ultima_data_tri].copy()


# 2. TRATAMENTO E AGREGAÇÃO (Média Ponderada)
# Garantir que as colunas numéricas estão no formato certo e remover nulos (ex: FIIs de Papel)
df_imoveis_atual['Area'] = pd.to_numeric(df_imoveis_atual['Area'], errors='coerce')
df_imoveis_atual['Percentual_Vacancia'] = pd.to_numeric(df_imoveis_atual['Percentual_Vacancia'], errors='coerce')
df_imoveis_atual['Percentual_Inadimplencia'] = pd.to_numeric(df_imoveis_atual['Percentual_Inadimplencia'], errors='coerce')

# Filtramos os imóveis que possuem os dados preenchidos
df_imoveis_validos = df_imoveis_atual.dropna(subset=['Area', 'Percentual_Vacancia']).copy()

# Criar a coluna do valor absoluto da vacância (Vacância * Peso)
df_imoveis_validos['Vacancia_Area_Absoluta'] = df_imoveis_validos['Percentual_Vacancia'] * df_imoveis_validos['Area']

# Agrupar somando as áreas totais e as vacâncias absolutas por Fundo
df_agregado = df_imoveis_validos.groupby('CNPJ_Fundo_Classe').agg({
    'Vacancia_Area_Absoluta': 'sum',
    'Area': 'sum',
    'Percentual_Inadimplencia': 'mean' # Inadimplência segue na média simples por enquanto
}).reset_index()

# Dividir a Vacância Absoluta pela Área Total para achar o Percentual Ponderado final
df_agregado['Percentual_Vacancia'] = df_agregado['Vacancia_Area_Absoluta'] / df_agregado['Area']

# Isolar as colunas limpas
df_vacancia_fundo = df_agregado[['CNPJ_Fundo_Classe', 'Percentual_Vacancia', 'Percentual_Inadimplencia']]


# 3. O MERGE (Unindo Contabilidade e Física)
# Assumindo que o df_micro (com LTV e VPA) já está na memória
df_matriz_quase_pronta = pd.merge(df_micro, df_vacancia_fundo, on='CNPJ_Fundo_Classe', how='left')

# Tratamento de Valores Nulos (O Ponto Cego da CVM)
# FIIs de Papel não reportam vacância física, então o merge vai gerar NaN.
# Precisamos preencher com zero para que o algoritmo de Machine Learning consiga calcular a distância.
df_matriz_quase_pronta['Percentual_Vacancia'] = df_matriz_quase_pronta['Percentual_Vacancia'].fillna(0)
df_matriz_quase_pronta['Percentual_Inadimplencia'] = df_matriz_quase_pronta['Percentual_Inadimplencia'].fillna(0)

print("\n--- Matriz Microeconômica Consolidada ---")
print(df_matriz_quase_pronta.head(70))


--- Matriz Microeconômica Consolidada ---
     CNPJ_Fundo_Classe Data_Referencia   Valor_Ativo  Patrimonio_Liquido  \
0   00.332.266/0001-31      2025-12-01  2.583944e+08        2.579699e+08   
1   00.613.094/0001-74      2025-12-01  1.970381e+08       -2.502495e+07   
2   00.762.723/0001-28      2025-12-01  9.744736e+07        9.585189e+07   
3   00.868.235/0001-08      2025-12-01  1.241421e+08        1.216396e+08   
4   01.201.140/0001-90      2025-12-01  5.255968e+08        5.217479e+08   
..                 ...             ...           ...                 ...   
65  11.769.604/0001-13      2025-12-01  1.922764e+09        1.304610e+09   
66  11.827.568/0001-05      2025-12-01  5.047303e+07        5.035478e+07   
67  11.839.593/0001-09      2025-12-01  6.363438e+09        5.497692e+09   
68  11.945.604/0001-27      2025-12-01  2.488960e+06        2.388743e+06   
69  12.005.956/0001-65      2025-12-01  5.044952e+09        4.602846e+09   

            VPA    DY_Mes  Total_Passivo    

In [5]:
#Varedura de dados
# 1. Checagem de Valores Nulos (NaNs)
print("--- 1. VALORES NULOS (NaNs) ---")
nulos = df_matriz_quase_pronta.isna().sum()
print(nulos[nulos > 0])
if nulos.sum() == 0:
    print(" Nenhum valor nulo encontrado. Merge perfeito!")

# 2. Resumo Estatístico (O radar de Outliers)
print("\n--- 2. ESTATÍSTICAS DESCRITIVAS (Distribuição) ---")
colunas_numericas = ['LTV', 'VPA', 'DY_Mes', 'Percentual_Vacancia', 'Percentual_Inadimplencia']
print(df_matriz_quase_pronta[colunas_numericas].describe().round(4))

# 3. Alarmes de Regras de Negócio ( números bizarros)
print("\n--- 3. ALARMES DE ANOMALIAS ---")

# A) Vacância impossível (> 100% ou < 0%)
vacancia_estranha = df_matriz_quase_pronta[
    (df_matriz_quase_pronta['Percentual_Vacancia'] > 1) | 
    (df_matriz_quase_pronta['Percentual_Vacancia'] < 0)
]
print(f"⚠️ Fundos com Vacância > 1 ou < 0: {len(vacancia_estranha)}")
if len(vacancia_estranha) > 0:
    print(vacancia_estranha[['CNPJ_Fundo_Classe', 'Percentual_Vacancia']].head())

# B) LTV Negativo (Passivo ou Ativo negativo, o que é um erro contábil)
ltv_negativo = df_matriz_quase_pronta[df_matriz_quase_pronta['LTV'] < 0]
print(f"⚠️ Fundos com LTV negativo: {len(ltv_negativo)}")
if len(ltv_negativo) > 0:
    print(ltv_negativo[['CNPJ_Fundo_Classe', 'LTV']].head())

# C) LTV > 1 (Alavancagem Extrema / PL Negativo)
# Isso é possível na realidade (vimos um no seu print anterior), mas é bom isolar para entender o risco.
ltv_estourado = df_matriz_quase_pronta[df_matriz_quase_pronta['LTV'] > 1]
print(f"⚠️ Fundos com LTV > 1 (Distress): {len(ltv_estourado)}")

# D) VPA Negativo ou Zerado
vpa_estranho = df_matriz_quase_pronta[df_matriz_quase_pronta['VPA'] <= 0]
print(f"⚠️ Fundos com VPA <= 0: {len(vpa_estranho)}")

--- 1. VALORES NULOS (NaNs) ---
DY_Mes    60
LTV        3
dtype: int64

--- 2. ESTATÍSTICAS DESCRITIVAS (Distribuição) ---
             LTV           VPA     DY_Mes  Percentual_Vacancia  \
count  1222.0000  1.225000e+03  1165.0000            1225.0000   
mean      1.8461  5.583532e+04     0.0104               0.0610   
std      41.8249  8.413971e+05     0.0806               0.2049   
min      -1.6530 -4.415749e+04    -0.8566               0.0000   
25%       0.0021  1.839600e+01     0.0000               0.0000   
50%       0.0132  1.005510e+02     0.0000               0.0000   
75%       0.1255  7.038593e+02     0.0074               0.0000   
max    1166.0246  2.536967e+07     1.3744               1.0000   

       Percentual_Inadimplencia  
count                 1225.0000  
mean                     0.0078  
std                      0.0677  
min                     -0.0592  
25%                      0.0000  
50%                      0.0000  
75%                      0.0000  
max       

In [6]:
#Limpeza
# 1. Cópia de segurança
df_limpo = df_matriz_quase_pronta.copy()

# 2. Tratamento de NaNs
# Se não tem DY no mês, assume-se 0 de distribuição
df_limpo['DY_Mes'] = df_limpo['DY_Mes'].fillna(0)
# Dropa os 3 fundos sem LTV (não dá para clusterizar estrutura de capital sem LTV)
df_limpo = df_limpo.dropna(subset=['LTV'])

# 3. Poda de LTV (Regra de Negócio: 0 <= LTV <= 1.5)
# LTV = 1.5 significa passivo 50% maior que o ativo (distress severo, mas possível).
df_limpo = df_limpo[(df_limpo['LTV'] >= 0) & (df_limpo['LTV'] <= 1.5)]

# 4. Poda de Dividend Yield (Regra de Negócio: 0 <= DY <= 0.05)
# Corta amortizações bizarras e ajustes contábeis negativos
df_limpo = df_limpo[(df_limpo['DY_Mes'] >= 0) & (df_limpo['DY_Mes'] <= 0.05)]

# 5. Poda de VPA (Regra de Negócio: VPA > 0 e limitando aberrações de milhões)
# Corta fundos com PL negativo e fundos de 1 cota institucional
df_limpo = df_limpo[(df_limpo['VPA'] > 0) & (df_limpo['VPA'] < 50000)]

# 6. Avaliação da Perda de Dados
fundos_antes = len(df_matriz_quase_pronta)
fundos_depois = len(df_limpo)
perda = fundos_antes - fundos_depois

print(f"✅ Limpeza concluída!")
print(f"Fundos iniciais: {fundos_antes}")
print(f"Fundos após filtro: {fundos_depois}")
print(f"Total de distorções e outliers removidos: {perda} fundos.")

# Verifica se sobrou algum erro
print("\nNovas estatísticas máximas após a limpeza:")
print(df_limpo[['LTV', 'DY_Mes', 'VPA']].max())

✅ Limpeza concluída!
Fundos iniciais: 1225
Fundos após filtro: 1114
Total de distorções e outliers removidos: 111 fundos.

Novas estatísticas máximas após a limpeza:
LTV           1.000000
DY_Mes        0.049678
VPA       43620.748901
dtype: float64


In [8]:
#Aqui é feito o cruzamento dos dados da base bruta, para conectar o ticket do fundo com a base via ISIN( que nada mais é do que um id do fundo)
# 1. CARREGAR A BASE GERAL DA CVM (Onde está o CNPJ e o ISIN)
df_geral_cvm = pd.read_csv('inf_mensal_fii_geral_2025.csv', sep=';', encoding='latin1')

# 2. EXTRAIR O TICKER DO ISIN NA CVM
# O ISIN brasileiro é BR + TICKER + TIPO (ex: BRADSHCTF005). 
# O Ticker são os 4 caracteres após o 'BR'.
def extrair_ticker_isin(isin):
    if pd.isna(isin) or len(str(isin)) < 6:
        return None
    return str(isin)[2:6].upper()

df_geral_cvm['Ticker_CVM'] = df_geral_cvm['Codigo_ISIN'].apply(extrair_ticker_isin)

# 3. CARREGAR A BASE DA B3 
df_b3 = pd.read_csv('fundosListados.csv', sep=';', encoding='latin1')
df_b3['Ticker_B3'] = df_b3['Fundo'].astype(str).str.strip().str.upper()

# 4. O MERGE PELO TICKER 
df_dicionario = pd.merge(
    df_geral_cvm[['CNPJ_Fundo_Classe', 'Ticker_CVM']].dropna(),
    df_b3[['Ticker_B3']],
    left_on='Ticker_CVM',
    right_on='Ticker_B3',
    how='inner'
).drop_duplicates(subset='Ticker_B3')

# 5. UNIR COM A MATRIZ DE FUNDAMENTOS LIMPA
df_final_investivel = pd.merge(
    df_limpo, 
    df_dicionario[['CNPJ_Fundo_Classe', 'Ticker_B3']], 
    on='CNPJ_Fundo_Classe', 
    how='inner'
)

# Ajuste final para o Yahoo Finance
df_final_investivel['Ticker_YF'] = df_final_investivel['Ticker_B3'] + '11.SA'

print(f" Amostra expandida para {len(df_final_investivel)} fundos.")
print(df_final_investivel[['Ticker_YF', 'LTV', 'Percentual_Vacancia']].head(10))

 Amostra expandida para 399 fundos.
   Ticker_YF       LTV  Percentual_Vacancia
0  FVPQ11.SA  0.001643             0.103933
1  CTXT11.SA  0.016373             1.000000
2  ABCP11.SA  0.007323             0.012190
3  FTCE11.SA  0.200288             0.361283
4  FMOF11.SA  0.046206             0.041700
5  RTEL11.SA  0.036910             0.236553
6  FPAB11.SA  0.034359             0.236117
7  SHPH11.SA  0.005005             0.005464
8  RCRB11.SA  0.114576             0.009509
9  HCRI11.SA  0.013911             0.000000


In [9]:
#Aqui eu quis dar uma olhada em todos os fundos do dataset, e depois aplciar mais uma filtragem.
# 1. Configurar o Pandas para mostrar tudo
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 2. Exibir a lista ordenada pelo Ticker para facilitar a conferência
print("--- LISTA COMPLETA DE FIIs INVESTÍVEIS (N=399) ---")
df_ordenado = df_final_investivel.sort_values('Ticker_YF')

# Exibe o Ticker, CNPJ e os fundamentos principais
print(df_ordenado[['Ticker_YF', 'CNPJ_Fundo_Classe', 'LTV', 'Percentual_Vacancia', 'VPA']])

# 3. Opcional: Salvar em Excel/CSV para a equipe analisar fora do Python
# df_ordenado.to_csv('matriz_micro_consolidada.csv', sep=';', index=False)
# print("\n✅ Arquivo 'matriz_micro_consolidada.csv' gerado para análise da equipe.")

--- LISTA COMPLETA DE FIIs INVESTÍVEIS (N=399) ---
     Ticker_YF   CNPJ_Fundo_Classe       LTV  Percentual_Vacancia           VPA
2    ABCP11.SA  01.201.140/0001-90  0.007323             0.012190    110.796096
394  ADSH11.SA  63.375.304/0001-53  0.490996             0.014000      9.700845
371  AERO11.SA  57.927.297/0001-52  0.215324             0.000000     99.729479
227  AFHI11.SA  36.642.293/0001-58  0.000934             0.000000     94.559785
209  AIEC11.SA  35.765.826/0001-26  0.022024             0.000000     76.573358
314  AJFI11.SA  51.472.985/0001-99  0.028331             0.066079     11.112882
15   ALMI11.SA  07.122.725/0001-00  0.024885             0.468000   2062.462425
143  ALZR11.SA  28.737.771/0001-85  0.250443             0.112501     10.640670
17   ANCR11.SA  07.789.135/0001-27  0.068858             0.042123    120.678243
268  APTO11.SA  41.081.356/0001-84  0.014313             0.000000      8.508690
288  APXM11.SA  43.010.485/0001-07  0.001352             1.000000    

In [10]:
#Aqui apliquei uam série de filtragens.

print("Análise de consistencia (N=399)")


# 1. Verificação de Escala de Cota (VPA vs Realidade)
# FIIs listados quase sempre valem entre R$ 5,00 e R$ 2.000,00.
# Valores fora disso indicam erros de base ou fundos extremamente ilíquidos.
vpa_suspeito = df_final_investivel[(df_final_investivel['VPA'] < 1) | (df_final_investivel['VPA'] > 5000)]
print(f" 1. Fundos com VPA fora da escala comercial (R$1 - R$5000): {len(vpa_suspeito)}")
if len(vpa_suspeito) > 0:
    print(vpa_suspeito[['Ticker_YF', 'VPA']].head(10))

# 2. Verificação de Conflito LTV vs Vacância
# Fundos de Papel (CRI) NÃO podem ter vacância física. 
# Se um fundo tem Vacância > 0 e é puramente de papel, há erro de classificação na CVM.
papel_com_vacancia = df_final_investivel[(df_final_investivel['Percentual_Vacancia'] > 0) & 
                                         (df_final_investivel['Ticker_YF'].str.contains('CR|CP|KN'))] # Tickers comuns de papel
print(f"2. Possíveis fundos de Papel com Vacância Física detectada: {len(papel_com_vacancia)}")

# 3. Análise de Outliers Multivariados (Z-Score)
# Vamos ver quem está a mais de 3 desvios padrão da média em LTV ou Vacância
from scipy import stats
cols_analise = ['LTV', 'Percentual_Vacancia']
z_scores = np.abs(stats.zscore(df_final_investivel[cols_analise]))
outliers_bolsa = df_final_investivel[(z_scores > 3).any(axis=1)]
print(f"3. Outliers Estatísticos detectados (Z-Score > 3): {len(outliers_bolsa)}")
if len(outliers_bolsa) > 0:
    print(outliers_bolsa[['Ticker_YF', 'LTV', 'Percentual_Vacancia']].head(10))

# 4. Verificação de Duplicidade de CNPJ
duplicados = df_final_investivel.duplicated(subset=['CNPJ_Fundo_Classe']).sum()
print(f"\n✅ 4. CNPJs duplicados na base: {duplicados}")

print("\n=========================================================")

Análise de consistencia (N=399)
 1. Fundos com VPA fora da escala comercial (R$1 - R$5000): 11
     Ticker_YF           VPA
62   KNRE11.SA      0.632653
97   ESTQ11.SA      0.014474
100  LOFT11.SA      0.121781
135  DLMT11.SA      0.515130
137  BPLC11.SA   9977.520587
254  SNLG11.SA      0.383496
299  AROA11.SA      0.961006
317  KLOG11.SA      0.947811
369  KPMR11.SA  10308.907804
372  KPRP11.SA      0.985339
2. Possíveis fundos de Papel com Vacância Física detectada: 9
3. Outliers Estatísticos detectados (Z-Score > 3): 28
     Ticker_YF       LTV  Percentual_Vacancia
1    CTXT11.SA  0.016373             1.000000
13   FAMB11.SA  0.183583             0.980100
38   PRSV11.SA  0.005934             1.000000
54   RBOP11.SA  0.035304             0.719200
82   TSNC11.SA  0.009993             1.000000
88   SAIC11.SA  0.029564             1.000000
91   FIVN11.SA  0.106260             0.853200
100  LOFT11.SA  0.362414             0.999382
118  BRIM11.SA  0.664623             0.000000
136  RECR1

In [ ]:
#Aplicando o filtro
# 1. Poda de VPA (Removendo os 11 fundos de centavos/milhares)
df_micro_final= df_final_investivel[
    (df_final_investivel['VPA'] >= 1) & 
    (df_final_investivel['VPA'] <= 5000)
].copy()

# 2. Poda de Vacância Extrema (Opcional, mas recomendado para clareza dos clusters)
# Vamos remover quem tem vacância de 100%, pois são casos de falência operacional
df_micro_final = df_micro_final[df_micro_final['Percentual_Vacancia'] < 0.95]

# 3. Resultado Final
fundos_finais = len(df_micro_final)
print(f"Sobraram {fundos_finais} fundos consistentes.")
print(df_micro_final.head())

#Exportando o csv
df_micro_final.to_csv('df_micro_final.csv', index=True)

In [ ]:
df = pd.read_csv('dados_macro_brutos.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

def acumular_cdi_mensal(df):
    df_mensal_ipca = df[['IPCA_Mensal']].resample('ME').mean()
    
    imab_preco = df[['IMAB11_Preco']].resample('ME').last()
    df_mensal_imab = imab_preco.pct_change() * 100
    df_mensal_imab.columns = ['IMAB_ret']

    cdi_mensal = (
        df[['CDI_Diario']]
        .apply(lambda x: (1 + x / 100))
        .resample('ME')
        .prod()
        .subtract(1)
        .multiply(100)
        .rename(columns={'CDI_Diario': 'CDI_Mensal'})
    )

    df_mensal = cdi_mensal.join(df_mensal_ipca).join(df_mensal_imab)
    df_mensal = df_mensal.dropna()
    return df_mensal

df_mensal = acumular_cdi_mensal(df)
df_mensal = df_mensal[df_mensal.index <= '2025-12-31']

df_mensal.info()
df_mensal.describe()
df_mensal.to_csv('dados_macro_tratados.csv')

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 71 entries, 2020-02-29 to 2025-12-31
Freq: ME
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CDI_Mensal   71 non-null     float64
 1   IPCA_Mensal  71 non-null     float64
 2   IMAB_ret     71 non-null     float64
dtypes: float64(3)
memory usage: 2.2 KB


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


In [ ]:
# 1. CARREGAMENTO


def carregar_base_macro(caminho="dados_macro_tratados.csv"):
    # Lê o CSV e valida as colunas esperadas
    df = pd.read_csv(caminho, index_col=0, parse_dates=True).sort_index()

    ausentes = {"CDI_Mensal", "IPCA_Mensal", "IMAB_ret"} - set(df.columns)
    if ausentes:
        raise ValueError(f"Colunas ausentes: {ausentes}")

    print(f"Base carregada: {len(df)} meses | {df.index[0].date()} → {df.index[-1].date()}")
    return df[["CDI_Mensal", "IPCA_Mensal", "IMAB_ret"]].dropna()



# 2. ORTOGONALIZAÇÃO VIA OLS

def _regressao_ols(y, X, titulo):
    # Roda OLS com constante e retorna resíduos
    res = sm.OLS(y, sm.add_constant(X)).fit()
    print(f"\nRegressão: {titulo}")
    print(res.summary())
    return res.resid, res


def ortogonalizar(df):
    # Passo 1: extrai choque puro de inflação (IPCA livre do CDI)
    u1, mod_ipca = _regressao_ols(
        df["IPCA_Mensal"], df[["CDI_Mensal"]],
        "IPCA_Mensal ~ CDI_Mensal"
    )

    # Passo 2: extrai choque puro de juro real (IMAB livre do CDI e IPCA)
    u2, mod_imab = _regressao_ols(
        df["IMAB_ret"], df[["CDI_Mensal", "IPCA_Mensal"]],
        "IMAB_ret ~ CDI_Mensal + IPCA_Mensal"
    )

    # Monta DataFrame com os vetores ortogonais
    df_ort = pd.DataFrame({
        "CDI_Mensal": df["CDI_Mensal"],
        "u1_IPCA"   : u1,
        "u2_IMAB"   : u2,
    }, index=df.index)

    return df_ort, {"IPCA": mod_ipca, "IMAB": mod_imab}



# 3. VALIDAÇÃO


def validar_ortogonalidade(df_ort):
    # Testa correlação entre os vetores — devem ser próximas de zero
    pares = [
        ("CDI_Mensal", "u1_IPCA"),
        ("CDI_Mensal", "u2_IMAB"),
        ("u1_IPCA",    "u2_IMAB"),
    ]

    print("\nTeste de ortogonalidade:")
    for v1, v2 in pares:
        r, p = stats.pearsonr(df_ort[v1], df_ort[v2])
        status = "OK" if abs(r) < 0.05 else "REVISAR"
        print(f"  {v1} x {v2}: r={r:.6f}  p={p:.6f}  [{status}]")

    return df_ort.corr()


# ------------------------------------------------------------
# 4. VISUALIZAÇÃO
# ------------------------------------------------------------

def plotar_diagnostico(df_original, df_ort, corr_antes, corr_depois,
                       salvar_em="diagnostico_ortogonalizacao.png"):
    # Painel 2x2: séries originais, vetores ortogonais, heatmaps antes/depois
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    def normalizar(d):
        return (d - d.mean()) / d.std()

    # Séries originais normalizadas
    norm = normalizar(df_original)
    for col in norm.columns:
        axes[0, 0].plot(norm.index, norm[col], label=col)
    axes[0, 0].set_title("Séries Originais (normalizadas)")
    axes[0, 0].legend(fontsize=7)
    axes[0, 0].axhline(0, linewidth=0.5, linestyle="--", color="gray")

    # Vetores ortogonalizados normalizados
    norm_ort = normalizar(df_ort)
    for col in norm_ort.columns:
        axes[0, 1].plot(norm_ort.index, norm_ort[col], label=col)
    axes[0, 1].set_title("Vetores Ortogonalizados (normalizados)")
    axes[0, 1].legend(fontsize=7)
    axes[0, 1].axhline(0, linewidth=0.5, linestyle="--", color="gray")

    # Heatmap correlação antes
    sns.heatmap(corr_antes.round(3), annot=True, fmt=".3f",
                cmap="RdYlGn", vmin=-1, vmax=1, ax=axes[1, 0])
    axes[1, 0].set_title("Correlação ANTES")

    # Heatmap correlação depois
    sns.heatmap(corr_depois.round(6), annot=True, fmt=".6f",
                cmap="RdYlGn", vmin=-1, vmax=1, ax=axes[1, 1])
    axes[1, 1].set_title("Correlação DEPOIS")

    plt.tight_layout()
    plt.savefig(salvar_em, dpi=150)
    plt.close()
    print(f"Gráfico salvo: {salvar_em}")



# 5. EXECUÇÃO


# Carrega a base macro tratada
df = carregar_base_macro("dados_macro_tratados.csv")

# Correlação antes da ortogonalização
corr_antes = df.corr()
print(f"\nCorrelação antes:\n{corr_antes.round(4)}")

# Ortogonaliza as variáveis macro
df_ort, modelos = ortogonalizar(df)

# Valida a ortogonalidade dos vetores resultantes
corr_depois = validar_ortogonalidade(df_ort)

# Gera painel de diagnóstico
plotar_diagnostico(df, df_ort, corr_antes, corr_depois)

# Salva os vetores ortogonais
df_ort.to_csv("macro_ortogonalizada.csv")
print(f"\nArquivo salvo: macro_ortogonalizada.csv | shape: {df_ort.shape}")

Base carregada: 71 meses | 2020-02-29 → 2025-12-31

Correlação antes:
             CDI_Mensal  IPCA_Mensal  IMAB_ret
CDI_Mensal       1.0000      -0.2593    0.0953
IPCA_Mensal     -0.2593       1.0000    0.0743
IMAB_ret         0.0953       0.0743    1.0000

Regressão: IPCA_Mensal ~ CDI_Mensal
                            OLS Regression Results                            
Dep. Variable:            IPCA_Mensal   R-squared:                       0.067
Model:                            OLS   Adj. R-squared:                  0.054
Method:                 Least Squares   F-statistic:                     4.974
Date:                Sat, 11 Apr 2026   Prob (F-statistic):             0.0290
Time:                        17:13:06   Log-Likelihood:                -37.383
No. Observations:                  71   AIC:                             78.77
Df Residuals:                      69   BIC:                             83.29
Df Model:                           1                                    

In [39]:
"""
Etapa 3 — Extração de Betas via Regressão de Séries Temporais
Lê:  macro_ortogonalizada.csv
Gera: retornos_fiis_mensais.csv | betas_macroeconomicos.csv | fiis_excluidos.csv
"""

import os
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# UNIVERSO DE FIIs E PARÂMETROS
# ------------------------------------------------------------

TICKERS_FIIS = [
    "FVPQ11.SA", "ABCP11.SA", "FTCE11.SA", "FMOF11.SA", "RTEL11.SA",
    "FPAB11.SA", "SHPH11.SA", "RCRB11.SA", "HCRI11.SA", "FLMA11.SA",
    "TRNT11.SA", "EURO11.SA", "EDFO11.SA", "ALMI11.SA", "SHDP11.SA",
    "ANCR11.SA", "NSLU11.SA", "HGBS11.SA", "VERE11.SA", "HUSC11.SA",
    "HTMX11.SA", "BRCR11.SA", "RBRD11.SA", "HGRE11.SA", "WPLZ11.SA",
    "SEQR11.SA", "CJCT11.SA", "FLRP11.SA", "DOVL11.SA", "PQDP11.SA",
    "CXCE11.SA", "STRX11.SA", "HGCR11.SA", "FAED11.SA", "HGPO11.SA",
    "MAXR11.SA", "FCFL11.SA", "VRTA11.SA", "HGLG11.SA", "GSFI11.SA",
    "BTLG11.SA", "RBDS11.SA", "KNRI11.SA", "VINO11.SA", "BBRC11.SA",
    "FISC11.SA", "ATSA11.SA", "CXTL11.SA", "PQAG11.SA", "ELDO11.SA",
    "JSRE11.SA", "RBOP11.SA", "CNES11.SA", "CARE11.SA", "VLJS11.SA",
    "PRSN11.SA", "PLRI11.SA", "FIIB11.SA", "BMLC11.SA", "ATCR11.SA",
    "VPSI11.SA", "NEWU11.SA", "RNGO11.SA", "VTPL11.SA", "EDGA11.SA",
    "BRHT11.SA", "VJFD11.SA", "SPTW11.SA", "BNFS11.SA", "RBVA11.SA",
    "CEOC11.SA", "FISD11.SA", "TRBL11.SA", "KNCR11.SA", "XPCM11.SA",
    "REIT11.SA", "MFII11.SA", "HRDF11.SA", "NVHO11.SA", "CXRI11.SA",
    "CBOP11.SA", "PORD11.SA", "FPNG11.SA", "VISC11.SA", "FIGS11.SA",
    "FIVN11.SA", "VOTS11.SA", "HFOF11.SA", "BTHI11.SA", "SPAF11.SA",
    "FINF11.SA", "CPTS11.SA", "RSPD11.SA", "BTWR11.SA", "BCIA11.SA",
    "BRCO11.SA", "BVAR11.SA", "RBRP11.SA", "VTPA11.SA", "VTVI11.SA",
    "NVIF11.SA", "SOLR11.SA", "BCRI11.SA", "SHOP11.SA", "MTOF11.SA",
    "VVCR11.SA", "MCCI11.SA", "GTWR11.SA", "VSHO11.SA", "JTPR11.SA",
    "BRIM11.SA", "PCAS11.SA", "VILG11.SA", "KNIP11.SA", "TGAR11.SA",
    "OUJP11.SA", "XPLG11.SA", "GGRC11.SA", "DAMT11.SA", "TEPP11.SA",
    "PNPR11.SA", "TCPF11.SA", "HAAA11.SA", "HLOG11.SA", "SPVJ11.SA",
    "SJAU11.SA", "RCFA11.SA", "RZAT11.SA", "XPCI11.SA", "XPIN11.SA",
    "TRXF11.SA", "WTSP11.SA", "ALZR11.SA", "XPML11.SA", "RBTS11.SA",
    "RBRR11.SA", "HGRU11.SA", "VGIR11.SA", "HUSI11.SA", "PATC11.SA",
    "KFOF11.SA", "KNHY11.SA", "RBRY11.SA", "TORD11.SA", "LASC11.SA",
    "HCTR11.SA", "FATN11.SA", "HABT11.SA", "LVBI11.SA", "RBHG11.SA",
    "VPPR11.SA", "JPPA11.SA", "VXXV11.SA", "XPSF11.SA", "HCST11.SA",
    "RBIR11.SA", "ERPA11.SA", "MOFF11.SA", "JCDA11.SA", "CRFF11.SA",
    "HOFC11.SA", "PBLV11.SA", "ARRI11.SA", "CACR11.SA", "RECT11.SA",
    "KEVE11.SA", "VIDS11.SA", "NEWL11.SA", "HSML11.SA", "HSLG11.SA",
    "BPML11.SA", "FLCR11.SA", "RFOF11.SA", "HOSI11.SA", "BLMG11.SA",
    "TCIN11.SA", "VGIP11.SA", "URPR11.SA", "HCHG11.SA", "INLG11.SA",
    "RDCI11.SA", "PEMA11.SA", "INRD11.SA", "TSER11.SA", "HILG11.SA",
    "BRIP11.SA", "BLMO11.SA", "HSAF11.SA", "HREC11.SA", "BZEL11.SA",
    "HPDP11.SA", "PVBI11.SA", "HDOF11.SA", "SEED11.SA", "RCFF11.SA",
    "RBRL11.SA", "JCDB11.SA", "PATL11.SA", "AIEC11.SA", "ZIFI11.SA",
    "KNSC11.SA", "TELD11.SA", "VSLH11.SA", "ITIT11.SA", "ITIP11.SA",
    "TRXB11.SA", "YUFI11.SA", "VIUR11.SA", "RZTR11.SA", "HBCR11.SA",
    "JFLL11.SA", "GLOG11.SA", "BTML11.SA", "CYCR11.SA", "RBRS11.SA",
    "RZAK11.SA", "AFHI11.SA", "SALI11.SA", "KISU11.SA", "VGHF11.SA",
    "BTSI11.SA", "DEVA11.SA", "RELG11.SA", "BBFO11.SA", "LPLP11.SA",
    "GARE11.SA", "CENU11.SA", "PNDL11.SA", "RBHY11.SA", "GCRI11.SA",
    "CVPR11.SA", "HGIC11.SA", "ROOF11.SA", "CXCO11.SA", "BICE11.SA",
    "HUCG11.SA", "TJKB11.SA", "LIFE11.SA", "DVFF11.SA", "SNFF11.SA",
    "FIIC11.SA", "IBCR11.SA", "NCRI11.SA", "VCRR11.SA", "CXAG11.SA",
    "SPMO11.SA", "ZAVI11.SA", "JASC11.SA", "EQIR11.SA", "IRIM11.SA",
    "SNCI11.SA", "BLCA11.SA", "AURB11.SA", "MMPD11.SA", "CSMC11.SA",
    "APTO11.SA", "VCRI11.SA", "RBRX11.SA", "LLAO11.SA", "MMVE11.SA",
    "WHGR11.SA", "GAME11.SA", "MGRI11.SA", "CFII11.SA", "CXCI11.SA",
    "JSAF11.SA", "KIVO11.SA", "KCRE11.SA", "DVLP11.SA", "KNPR11.SA",
    "KNHF11.SA", "HRES11.SA", "BIPD11.SA", "MANA11.SA", "DPRO11.SA",
    "SPXS11.SA", "CYLD11.SA", "CCME11.SA", "NMKS11.SA", "SPDE11.SA",
    "SNEL11.SA", "VDSV11.SA", "TRXY11.SA", "ARXD11.SA", "RINV11.SA",
    "OCRE11.SA", "ZAGH11.SA", "TELM11.SA", "MCEM11.SA", "PNRC11.SA",
    "GCOI11.SA", "LTMT11.SA", "BROF11.SA", "CLIN11.SA", "VVRI11.SA",
    "BLOG11.SA", "NAUI11.SA", "YEES11.SA", "AJFI11.SA", "SAPI11.SA",
    "BETW11.SA", "BIPE11.SA", "PMIS11.SA", "VRTM11.SA", "AZPL11.SA",
    "ZAVC11.SA", "KORE11.SA", "SNME11.SA", "CPLG11.SA", "IBBP11.SA",
    "INDE11.SA", "URHF11.SA", "VGII11.SA", "CCVA11.SA", "VCTH11.SA",
    "LSOI11.SA", "ICNE11.SA", "ARTE11.SA", "IVCI11.SA", "HGBL11.SA",
    "EMET11.SA", "KRES11.SA", "VGRI11.SA", "SMRE11.SA", "PATA11.SA",
    "EGDB11.SA", "DAMA11.SA", "RZZI11.SA", "RENV11.SA", "BBIG11.SA",
    "EAGL11.SA", "GRUL11.SA", "DVLT11.SA", "BTYU11.SA", "JSCR11.SA",
    "SPXL11.SA", "LRED11.SA", "QTZD11.SA", "SPXG11.SA", "KFEN11.SA",
    "HOMS11.SA", "AZPE11.SA", "GCDL11.SA", "RZLC11.SA", "CEBB11.SA",
    "RECM11.SA", "TOPP11.SA", "SPG211.SA", "SPXM11.SA", "MCLO11.SA",
    "SHIP11.SA", "SOFF11.SA", "AERO11.SA", "BGS111.SA", "GLPF11.SA",
    "GFDL11.SA", "SHPP11.SA", "EIRA11.SA", "GLCR11.SA", "MAGM11.SA",
    "CVFL11.SA", "RCRI11.SA", "CDHY11.SA", "KNGR11.SA", "KGUJ11.SA",
    "HJCT11.SA", "TRCO11.SA", "RBFY11.SA", "MCMV11.SA", "GSRF11.SA",
    "FRBC11.SA", "TVOI11.SA", "ADSH11.SA", "LSOP11.SA", "CLSM11.SA",
    "MIDW11.SA", "MXRF11.SA",
]

DATA_INICIO     = "2020-01-01"
DATA_FIM        = "2025-12-31"  # corte, sem vazamento de dados futuros
MIN_OBSERVACOES = 24            # mínimo de meses para entrar na regressão
BATCH_SIZE      = 50            # tamanho do lote de download



# 1. CARREGAMENTO DA MACRO ORTOGONALIZADA


def carregar_macro(caminho="macro_ortogonalizada.csv"):
    # Lê os vetores ortogonais e aplica o corte temporal
    df = pd.read_csv(caminho, index_col=0, parse_dates=True).sort_index()

    ausentes = {"CDI_Mensal", "u1_IPCA", "u2_IMAB"} - set(df.columns)
    if ausentes:
        raise ValueError(f"Colunas ausentes: {ausentes}")

    df = df[df.index <= DATA_FIM]
    print(f"Macro carregada: {len(df)} meses | {df.index[0].date()} → {df.index[-1].date()}")
    return df[["CDI_Mensal", "u1_IPCA", "u2_IMAB"]].dropna()



# 2. DOWNLOAD DOS RETORNOS


def _baixar_lote(tickers_lote, data_inicio, data_fim):
    # Baixa um lote de tickers e retorna preços mensais (último fechamento do mês)
    raw = yf.download(tickers_lote, start=data_inicio, end=data_fim,
                      progress=False, auto_adjust=True)["Close"]

    if isinstance(raw, pd.Series):
        raw = raw.to_frame(name=tickers_lote[0])
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    raw.index = raw.index.tz_localize(None)
    return raw.resample("ME").last()


def baixar_retornos(tickers, caminho_cache="retornos_fiis_mensais.csv"):
    # Usa cache local se disponível — evita re-download instável do yfinance
    if os.path.exists(caminho_cache):
        print(f"Retornos carregados do cache: {caminho_cache}")
        return pd.read_csv(caminho_cache, index_col=0, parse_dates=True)

    print(f"Baixando {len(tickers)} FIIs em lotes de {BATCH_SIZE}...")
    lotes = [tickers[i:i+BATCH_SIZE] for i in range(0, len(tickers), BATCH_SIZE)]
    lista = []

    for i, lote in enumerate(lotes, 1):
        print(f"  Lote {i}/{len(lotes)}...", end=" ")
        try:
            lista.append(_baixar_lote(lote, DATA_INICIO, DATA_FIM))
            print("ok")
        except Exception as e:
            print(f"erro: {e}")

    # Consolida lotes e calcula retorno mensal simples
    precos = pd.concat(lista, axis=1)
    precos = precos.loc[:, ~precos.columns.duplicated()]
    retornos = precos.pct_change().dropna(how="all")

    retornos.to_csv(caminho_cache)
    print(f"Retornos salvos: {caminho_cache} | shape: {retornos.shape}")
    return retornos



# 3. EXTRAÇÃO DE BETAS POR REGRESSÃO OLS


def extrair_betas(retornos, macro):
    # Alinha os índices e prepara a matrix de regressores
    idx = retornos.index.intersection(macro.index)
    X_base = sm.add_constant(macro.loc[idx])

    print(f"\nExtração de betas: {len(idx)} meses em comum | mínimo exigido: {MIN_OBSERVACOES}m\n")

    resultados, excluidos = [], []

    for ticker in retornos.columns:
        y = retornos.loc[idx, ticker].dropna()
        n = len(y)

        # Registra fundos sem dados ou com série insuficiente
        if n == 0:
            excluidos.append({"ticker": ticker, "obs": 0, "motivo": "sem dados",
                               "inicio": None, "fim": None})
            continue
        if n < MIN_OBSERVACOES:
            excluidos.append({"ticker": ticker, "obs": n,
                               "motivo": f"fundo recente ({n} meses < {MIN_OBSERVACOES})",
                               "inicio": y.index.min().strftime("%Y-%m"),
                               "fim":    y.index.max().strftime("%Y-%m")})
            continue

        # Roda OLS e armazena coeficientes, p-valores e qualidade do ajuste
        try:
            m = sm.OLS(y, X_base.loc[y.index]).fit()
            b = lambda k: m.params.get(k, np.nan)
            p = lambda k: m.pvalues.get(k, np.nan)
            t = lambda k: m.tvalues.get(k, np.nan)

            resultados.append({
                "ticker"   : ticker,
                "alpha"    : b("const"),
                "beta_CDI" : b("CDI_Mensal"),
                "beta_IPCA": b("u1_IPCA"),
                "beta_IMAB": b("u2_IMAB"),
                "t_CDI"    : t("CDI_Mensal"),
                "t_IPCA"   : t("u1_IPCA"),
                "t_IMAB"   : t("u2_IMAB"),
                "p_alpha"  : p("const"),
                "p_CDI"    : p("CDI_Mensal"),
                "p_IPCA"   : p("u1_IPCA"),
                "p_IMAB"   : p("u2_IMAB"),
                "R2"       : m.rsquared,
                "R2_adj"   : m.rsquared_adj,
                "obs"      : int(n),
                "inicio"   : y.index.min().strftime("%Y-%m"),
                "fim"      : y.index.max().strftime("%Y-%m"),
                "sig_CDI"  : p("CDI_Mensal") < 0.10,
                "sig_IPCA" : p("u1_IPCA")    < 0.10,
                "sig_IMAB" : p("u2_IMAB")    < 0.10,
            })

            print(f"  {ticker:<14} beta_CDI={b('CDI_Mensal'):+.3f}  "
                  f"beta_IPCA={b('u1_IPCA'):+.3f}  "
                  f"beta_IMAB={b('u2_IMAB'):+.3f}  "
                  f"R2adj={m.rsquared_adj:.3f}  n={n}")

        except Exception as e:
            excluidos.append({"ticker": ticker, "obs": n,
                               "motivo": f"erro: {e}",
                               "inicio": y.index.min().strftime("%Y-%m"),
                               "fim":    y.index.max().strftime("%Y-%m")})

    df_betas    = pd.DataFrame(resultados).set_index("ticker")
    df_excluidos = pd.DataFrame(excluidos).set_index("ticker") if excluidos else pd.DataFrame()

    print(f"\nBetas extraídos: {len(df_betas)} FIIs | Excluídos: {len(df_excluidos)} FIIs")
    return df_betas, df_excluidos



# 4. FILTRO DE OUTLIERS


def filtrar_outliers(df_betas, threshold=5.0, obs_minimo_protegido=48):
    # Remove fundos com betas absurdos usando Z-score robusto (mediana + MAD)
    cols = ["beta_CDI", "beta_IPCA", "beta_IMAB"]
    mediana = df_betas[cols].median()
    mad     = (df_betas[cols] - mediana).abs().median()
    zscore  = (df_betas[cols] - mediana) / (mad * 1.4826)

    # Critério: outlier extremo (z>10) OU outlier moderado com série curta
    is_absurdo  = (zscore.abs() > 10).any(axis=1)
    is_moderado = (zscore.abs() > threshold).any(axis=1)
    serie_curta = df_betas["obs"] < obs_minimo_protegido
    is_outlier  = is_absurdo | (is_moderado & serie_curta)

    df_limpo    = df_betas[~is_outlier].copy()
    df_outliers = df_betas[is_outlier].copy()

    print(f"\nFiltro de outliers: {len(df_betas)} → {len(df_limpo)} FIIs "
          f"({len(df_outliers)} removidos)")
    if not df_outliers.empty:
        print(df_outliers[cols + ["R2_adj", "obs"]].round(4).to_string())

    return df_limpo, df_outliers



# 5. VISUALIZAÇÃO


def plotar_betas(df_betas, df_excluidos, salvar_em="diagnostico_betas.png"):
    # Painel 2x2: distribuição dos betas, scatter, R2, cobertura temporal
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    # Histograma dos três betas sobrepostos
    for col, label in zip(["beta_CDI", "beta_IPCA", "beta_IMAB"],
                           ["beta CDI", "beta IPCA", "beta IMAB"]):
        axes[0, 0].hist(df_betas[col].dropna(), bins=25, alpha=0.6, label=label)
    axes[0, 0].axvline(0, linewidth=0.8, linestyle="--", color="gray")
    axes[0, 0].set_title(f"Distribuição dos Betas (N={len(df_betas)})")
    axes[0, 0].legend(fontsize=7)

    # Scatter beta_CDI x beta_IPCA colorido por beta_IMAB
    sc = axes[0, 1].scatter(df_betas["beta_CDI"], df_betas["beta_IPCA"],
                             c=df_betas["beta_IMAB"], cmap="RdBu_r", s=25, alpha=0.8)
    plt.colorbar(sc, ax=axes[0, 1], label="beta IMAB")
    axes[0, 1].axhline(0, linewidth=0.5, linestyle="--", color="gray")
    axes[0, 1].axvline(0, linewidth=0.5, linestyle="--", color="gray")
    axes[0, 1].set_xlabel("beta CDI")
    axes[0, 1].set_ylabel("beta IPCA")
    axes[0, 1].set_title("beta_CDI x beta_IPCA (cor = beta_IMAB)")

    # Histograma do R2 ajustado
    r2 = df_betas["R2_adj"].dropna()
    axes[1, 0].hist(r2, bins=25, alpha=0.8)
    axes[1, 0].axvline(r2.median(), linewidth=1.2, linestyle="--",
                        color="red", label=f"mediana={r2.median():.3f}")
    axes[1, 0].axvline(0, linewidth=0.8, linestyle=":", color="gray")
    axes[1, 0].set_title("Distribuição do R2 Ajustado")
    axes[1, 0].legend(fontsize=7)

    # Histograma de meses de cobertura por FII
    obs = df_betas["obs"].dropna()
    axes[1, 1].hist(obs, bins=25, alpha=0.8)
    axes[1, 1].axvline(MIN_OBSERVACOES, linewidth=1.2, linestyle="--",
                        color="red", label=f"minimo={MIN_OBSERVACOES}m")
    axes[1, 1].set_title(f"Cobertura Temporal ({len(df_excluidos)} FIIs excluídos)")
    axes[1, 1].legend(fontsize=7)

    plt.tight_layout()
    plt.savefig(salvar_em, dpi=150)
    plt.close()
    print(f"Gráfico salvo: {salvar_em}")



# 6. EXECUÇÃO


# Carrega a macro ortogonalizada
macro = carregar_macro("macro_ortogonalizada.csv")

# Baixa retornos (usa cache se já existir)
retornos = baixar_retornos(TICKERS_FIIS)

# Extrai betas por regressão OLS para cada FII
df_betas, df_excluidos = extrair_betas(retornos, macro)

# Remove fundos com betas fora de escala
df_betas, df_outliers = filtrar_outliers(df_betas)

# Gera painel de diagnóstico
plotar_betas(df_betas, df_excluidos)

# Salva os resultados
df_betas.to_csv("betas_macroeconomicos.csv")
df_excluidos.to_csv("fiis_excluidos.csv")
df_outliers.to_csv("betas_outliers_removidos.csv")

print(f"\nArquivos salvos:")
print(f"  betas_macroeconomicos.csv   ({len(df_betas)} FIIs)")
print(f"  fiis_excluidos.csv          ({len(df_excluidos)} FIIs)")
print(f"  betas_outliers_removidos.csv ({len(df_outliers)} FIIs)")

Macro carregada: 71 meses | 2020-02-29 → 2025-12-31
Retornos carregados do cache: retornos_fiis_mensais.csv

Extração de betas: 71 meses em comum | mínimo exigido: 24m

  ABCP11.SA      beta_CDI=+0.036  beta_IPCA=-0.007  beta_IMAB=+0.008  R2adj=0.075  n=71
  ALMI11.SA      beta_CDI=+0.013  beta_IPCA=-0.013  beta_IMAB=+0.004  R2adj=-0.023  n=60
  ANCR11.SA      beta_CDI=+0.103  beta_IPCA=-0.028  beta_IMAB=+0.017  R2adj=-0.011  n=71
  ATSA11.SA      beta_CDI=+0.007  beta_IPCA=+0.006  beta_IMAB=-0.002  R2adj=-0.020  n=71
  BBRC11.SA      beta_CDI=+0.037  beta_IPCA=-0.016  beta_IMAB=+0.006  R2adj=0.083  n=71
  BRCR11.SA      beta_CDI=+0.030  beta_IPCA=-0.020  beta_IMAB=+0.013  R2adj=0.157  n=71
  BTLG11.SA      beta_CDI=+0.008  beta_IPCA=-0.012  beta_IMAB=+0.012  R2adj=0.398  n=71
  CJCT11.SA      beta_CDI=+0.012  beta_IPCA=+0.020  beta_IMAB=-0.002  R2adj=-0.002  n=61
  CXTL11.SA      beta_CDI=+0.042  beta_IPCA=-0.006  beta_IMAB=+0.006  R2adj=-0.011  n=71
  DOVL11.SA      beta_CDI=+0.029  

In [40]:
def filtrar_outliers(df_betas, threshold=5.0, obs_minimo_protegido=48):
    # Remove fundos com betas absurdos usando Z-score robusto (mediana + MAD)
    cols = ["beta_CDI", "beta_IPCA", "beta_IMAB"]
    mediana = df_betas[cols].median()
    mad     = (df_betas[cols] - mediana).abs().median()
    zscore  = (df_betas[cols] - mediana) / (mad * 1.4826)

    # Outlier absurdo (z>10) removido sempre; outlier moderado só se série curta
    is_absurdo  = (zscore.abs() > 10).any(axis=1)
    is_moderado = (zscore.abs() > threshold).any(axis=1)
    serie_curta = df_betas["obs"] < obs_minimo_protegido
    is_outlier  = is_absurdo | (is_moderado & serie_curta)

    df_limpo    = df_betas[~is_outlier].copy()
    df_outliers = df_betas[is_outlier].copy()

    print(f"Filtro de outliers: {len(df_betas)} → {len(df_limpo)} FIIs ({len(df_outliers)} removidos)")
    if not df_outliers.empty:
        print(df_outliers[cols + ["R2_adj", "obs"]].round(4).to_string())

    return df_limpo, df_outliers

In [42]:
# Recarrega a versão 
df_betas = pd.read_csv("betas_macroeconomicos.csv", index_col=0)
df_betas_limpo, df_outliers = filtrar_outliers_betas(df_betas)
df_betas_limpo.to_csv("betas_macroeconomicos.csv")
df_outliers.to_csv("betas_outliers_removidos.csv")
print(f" Base final: {len(df_betas_limpo)} FIIs")


  FILTRO DE OUTLIERS — threshold=5.0 / proteção série >=48m
  FIIs antes do filtro : 222
  FIIs removidos       : 0
  FIIs aprovados       : 222
 Base final: 222 FIIs


In [43]:
df_betas_limpo.to_csv("betas_macroeconomicos.csv")
df_outliers.to_csv("betas_outliers_removidos.csv")

print(f"✅ Base final aprovada: {len(df_betas_limpo)} FIIs")
print(f"📋 Removidos documentados em: 'betas_outliers_removidos.csv'")

✅ Base final aprovada: 222 FIIs
📋 Removidos documentados em: 'betas_outliers_removidos.csv'


In [ ]:
# 1. CARREGAMENTO E INSPEÇÃO DA BASE MICRO


micro = pd.read_csv("df_micro_final.csv", index_col=0)

print("BASE MICRO")
print(f"  Shape     : {micro.shape}")
print(f"  Index     : {micro.index.name} | dtype: {micro.index.dtype}")
print(f"\n  Colunas e tipos:")
print(micro.dtypes.to_string())
print(f"\n  Amostra do índice (5 primeiros): {micro.index[:5].tolist()}")
print(f"\n  NaNs por coluna:")
print(micro.isna().sum().to_string())
print(f"\n  Estatísticas descritivas:")
print(micro.describe().round(4).to_string())



# 2. CARREGAMENTO E INSPEÇÃO DA BASE MACRO (BETAS)


macro = pd.read_csv("betas_macroeconomicos.csv", index_col=0)

print("\n\nBASE MACRO (BETAS)")
print(f"  Shape     : {macro.shape}")
print(f"  Index     : {macro.index.name} | dtype: {macro.index.dtype}")
print(f"\n  Colunas e tipos:")
print(macro.dtypes.to_string())
print(f"\n  Amostra do índice (5 primeiros): {macro.index[:5].tolist()}")
print(f"\n  NaNs por coluna:")
print(macro.isna().sum().to_string())
print(f"\n  Estatísticas descritivas:")
print(macro.describe().round(4).to_string())



# 3. DIAGNÓSTICO DE COMPATIBILIDADE DOS ÍNDICES


print("\n\nDIAGNÓSTICO DE COMPATIBILIDADE")

# Verifica formato dos tickers nos dois índices
print(f"\n  Exemplo micro : {micro.index[0]}")
print(f"  Exemplo macro : {macro.index[0]}")

# Interseção e diferenças
idx_micro  = set(micro.index)
idx_macro  = set(macro.index)
em_ambos   = idx_micro & idx_macro
so_micro   = idx_micro - idx_macro
so_macro   = idx_macro - idx_micro

print(f"\n  FIIs na base micro         : {len(idx_micro)}")
print(f"  FIIs na base macro         : {len(idx_macro)}")
print(f"  FIIs em ambas (merge)      : {len(em_ambos)}")
print(f"  Só na micro (sem betas)    : {len(so_micro)}")
print(f"  Só na macro (sem micro)    : {len(so_macro)}")

if so_macro:
    print(f"\n  Tickers só na macro (verificar): {sorted(so_macro)}")

if so_micro:
    print(f"\n  Primeiros 10 só na micro: {sorted(so_micro)[:10]}")

BASE MICRO
  Shape     : (377, 12)
  Index     : None | dtype: int64

  Colunas e tipos:
CNPJ_Fundo_Classe            object
Data_Referencia              object
Valor_Ativo                 float64
Patrimonio_Liquido          float64
VPA                         float64
DY_Mes                      float64
Total_Passivo               float64
LTV                         float64
Percentual_Vacancia         float64
Percentual_Inadimplencia    float64
Ticker_B3                    object
Ticker_YF                    object

  Amostra do índice (5 primeiros): [0, 2, 3, 4, 5]

  NaNs por coluna:
CNPJ_Fundo_Classe           0
Data_Referencia             0
Valor_Ativo                 0
Patrimonio_Liquido          0
VPA                         0
DY_Mes                      0
Total_Passivo               0
LTV                         0
Percentual_Vacancia         0
Percentual_Inadimplencia    0
Ticker_B3                   0
Ticker_YF                   0

  Estatísticas descritivas:
        Valor_Ativ

In [ ]:
# Carrega as bases
micro = pd.read_csv("df_micro_final.csv", index_col=0)
macro = pd.read_csv("betas_macroeconomicos.csv", index_col=0)

# Confirma que Ticker_YF tem o mesmo formato do índice da macro
print("Amostra Ticker_YF na micro:", micro["Ticker_YF"].head().tolist())
print("Amostra índice da macro    :", macro.index[:5].tolist())

# Define Ticker_YF como índice da micro para alinhar com a macro
micro = micro.set_index("Ticker_YF")

# Merge — inner: só FIIs presentes nas duas bases
# left:  todos da micro, NaN nos betas dos que não têm série histórica
base_quantamental = micro.join(macro, how="inner")

# Diagnóstico do resultado
print(f"\nMicro          : {len(micro)} FIIs")
print(f"Macro          : {len(macro)} FIIs")
print(f"Base final     : {len(base_quantamental)} FIIs")
print(f"Excluídos      : {len(micro) - len(base_quantamental)} FIIs (sem betas)")
print(f"\nShape final    : {base_quantamental.shape}")
print(f"NaNs totais    : {base_quantamental.isna().sum().sum()}")
print(f"\nColunas finais :\n{base_quantamental.dtypes.to_string()}")

# Salva
base_quantamental.to_csv("base_quantamental.csv")
print(f"\nArquivo salvo: base_quantamental.csv")

Amostra Ticker_YF na micro: ['FVPQ11.SA', 'ABCP11.SA', 'FTCE11.SA', 'FMOF11.SA', 'RTEL11.SA']
Amostra índice da macro    : ['ABCP11.SA', 'ALMI11.SA', 'ANCR11.SA', 'ATSA11.SA', 'BBRC11.SA']

Micro          : 377 FIIs
Macro          : 222 FIIs
Base final     : 222 FIIs
Excluídos      : 155 FIIs (sem betas)

Shape final    : (222, 30)
NaNs totais    : 0

Colunas finais :
CNPJ_Fundo_Classe            object
Data_Referencia              object
Valor_Ativo                 float64
Patrimonio_Liquido          float64
VPA                         float64
DY_Mes                      float64
Total_Passivo               float64
LTV                         float64
Percentual_Vacancia         float64
Percentual_Inadimplencia    float64
Ticker_B3                    object
alpha                       float64
beta_CDI                    float64
beta_IPCA                   float64
beta_IMAB                   float64
t_CDI                       float64
t_IPCA                      float64
t_IMAB           

In [ ]:
base = pd.read_csv("base_quantamental.csv", index_col=0)

# Remove colunas que não entram no HAC
# CNPJ e datas são identificadores, não features
# alpha, t_, p_, R2, sig_, obs, inicio, fim são métricas da regressão, não inputs do modelo
colunas_remover = [
    "CNPJ_Fundo_Classe", "Data_Referencia", "Ticker_B3",  # identificadores
    "alpha",                                                # intercepto da regressão
    "t_CDI", "t_IPCA", "t_IMAB",                          # t-stats (redundantes com betas)
    "p_alpha", "p_CDI", "p_IPCA", "p_IMAB",               # p-valores (redundantes)
    "R2", "R2_adj",                                        # qualidade do ajuste
    "sig_CDI", "sig_IPCA", "sig_IMAB",                    # flags booleanas
    "obs", "inicio", "fim",                                # metadados da regressão
]

base_hac = base.drop(columns=colunas_remover)

print(f"Shape para o HAC: {base_hac.shape}")
print(f"\nFeatures que entram no modelo:")
print(base_hac.dtypes.to_string())
print(f"\nEstatísticas descritivas:")
print(base_hac.describe().round(4).to_string())

base_hac.to_csv("base_hac.csv")
print("\nArquivo salvo: base_hac.csv")

Shape para o HAC: (222, 11)

Features que entram no modelo:
Valor_Ativo                 float64
Patrimonio_Liquido          float64
VPA                         float64
DY_Mes                      float64
Total_Passivo               float64
LTV                         float64
Percentual_Vacancia         float64
Percentual_Inadimplencia    float64
beta_CDI                    float64
beta_IPCA                   float64
beta_IMAB                   float64

Estatísticas descritivas:
        Valor_Ativo  Patrimonio_Liquido        VPA    DY_Mes  Total_Passivo       LTV  Percentual_Vacancia  Percentual_Inadimplencia  beta_CDI  beta_IPCA  beta_IMAB
count  2.220000e+02        2.220000e+02   222.0000  222.0000   2.220000e+02  222.0000             222.0000                  222.0000  222.0000   222.0000   222.0000
mean   9.370341e+08        8.245384e+08   159.2220    0.0071   1.134803e+08    0.0647               0.0563                    0.0234    0.0205    -0.0029     0.0059
std    1.598101e+09   

In [ ]:
base = pd.read_csv("base_hac.csv", index_col=0)

# Lista de tickers na base
tickers = base.index.tolist()

# Baixa o preço de fechamento de dezembro/2025
print(f"Baixando preços de dez/2025 para {len(tickers)} FIIs...")
precos_raw = yf.download(
    tickers,
    start="2025-12-01",
    end="2025-12-31",
    progress=False,
    auto_adjust=True
)["Close"]

# Normaliza colunas
if isinstance(precos_raw.columns, pd.MultiIndex):
    precos_raw.columns = precos_raw.columns.get_level_values(0)
precos_raw.index = precos_raw.index.tz_localize(None)

# Pega o último preço disponível de dezembro para cada ticker
preco_dez = precos_raw.last("1D").iloc[-1]
preco_dez.name = "preco_dez2025"

print(f"\nPreços obtidos: {preco_dez.notna().sum()} / {len(tickers)} tickers")
print(f"Sem preço     : {preco_dez.isna().sum()} tickers")
if preco_dez.isna().any():
    print(f"  {preco_dez[preco_dez.isna()].index.tolist()}")

# Calcula P/VPA
base["preco_dez2025"] = preco_dez
base["P_VPA"] = base["preco_dez2025"] / base["VPA"]

print(f"\nP/VPA calculado:")
print(base["P_VPA"].describe().round(4))
print(f"\nNaNs no P/VPA: {base['P_VPA'].isna().sum()}")

# Remove VPA bruto, preço e as colunas de tamanho absoluto
# Mantém só P/VPA no lugar de VPA
base_hac = base.drop(columns=["VPA", "preco_dez2025",
                                "Valor_Ativo", "Patrimonio_Liquido", "Total_Passivo"])

print(f"\nFeatures finais para o HAC:")
print(base_hac.dtypes.to_string())
print(f"\nShape: {base_hac.shape}")
print(base_hac.describe().round(4).to_string())

base_hac.to_csv("base_hac.csv")
print("\nArquivo salvo: base_hac.csv")

Baixando preços de dez/2025 para 222 FIIs...


$STRX11.SA: possibly delisted; no price data found  (1d 2025-12-01 -> 2025-12-31)

1 Failed download:
['STRX11.SA']: possibly delisted; no price data found  (1d 2025-12-01 -> 2025-12-31)



Preços obtidos: 215 / 222 tickers
Sem preço     : 7 tickers
  ['ATSA11.SA', 'BLMO11.SA', 'CJCT11.SA', 'ERPA11.SA', 'KEVE11.SA', 'STRX11.SA', 'TRNT11.SA']

P/VPA calculado:
count    215.0000
mean       0.8246
std        0.4760
min        0.0498
25%        0.6720
50%        0.8161
75%        0.9270
max        5.7547
Name: P_VPA, dtype: float64

NaNs no P/VPA: 7

Features finais para o HAC:
DY_Mes                      float64
LTV                         float64
Percentual_Vacancia         float64
Percentual_Inadimplencia    float64
beta_CDI                    float64
beta_IPCA                   float64
beta_IMAB                   float64
P_VPA                       float64

Shape: (222, 8)
         DY_Mes       LTV  Percentual_Vacancia  Percentual_Inadimplencia  beta_CDI  beta_IPCA  beta_IMAB     P_VPA
count  222.0000  222.0000             222.0000                  222.0000  222.0000   222.0000   222.0000  215.0000
mean     0.0071    0.0647               0.0563                    0.0234 

In [48]:
base_hac = pd.read_csv("base_hac.csv", index_col=0)

print(f"Antes: {len(base_hac)} FIIs | NaNs: {base_hac.isna().sum().sum()}")

# Remove os 7 FIIs sem preço em dez/2025
base_hac = base_hac.dropna()

print(f"Depois: {len(base_hac)} FIIs | NaNs: {base_hac.isna().sum().sum()}")
print(f"\nFIIs removidos: ATSA11, BLMO11, CJCT11, ERPA11, KEVE11, STRX11, TRNT11")

base_hac.to_csv("base_hac.csv")
print(f"\nArquivo salvo: base_hac.csv | shape final: {base_hac.shape}")
print(f"\nFeatures:")
print(base_hac.dtypes.to_string())

Antes: 222 FIIs | NaNs: 7
Depois: 215 FIIs | NaNs: 0

FIIs removidos: ATSA11, BLMO11, CJCT11, ERPA11, KEVE11, STRX11, TRNT11

Arquivo salvo: base_hac.csv | shape final: (215, 8)

Features:
DY_Mes                      float64
LTV                         float64
Percentual_Vacancia         float64
Percentual_Inadimplencia    float64
beta_CDI                    float64
beta_IPCA                   float64
beta_IMAB                   float64
P_VPA                       float64
